<div style="background:linear-gradient(135deg,#0d1117,#13243b,#0f3a5f);padding:44px 38px;border-radius:16px;color:#f0f6fc;font-family:'Segoe UI',sans-serif;border:1px solid #30363d;">
  <div style="font-size:.8em;letter-spacing:3px;opacity:.65;text-transform:uppercase;margin-bottom:10px;">Publication Figures · IEEE JBHI · Inference Only</div>
  <h1 style="font-size:2.0em;margin:0 0 12px 0;font-weight:700;line-height:1.25;color:#f0f6fc !important;">Yayın Kalitesinde Görüntü Panelleri</h1>
  <h2 style="font-size:1.05em;font-weight:300;opacity:.82;margin:0 0 22px 0;line-height:1.5;color:#f0f6fc !important;">IEEE kolon genişliğinde, vektör PDF çıktılı, 600 dpi gömülü raster</h2>
  <hr style="border:0;border-top:1px solid #30363d;margin:0 0 18px 0;">
  <div style="display:grid;grid-template-columns:1fr 1fr;gap:12px;font-size:.88em;opacity:.88;">
    <div><b>Eğitim:</b> yok — kayıtlı kontrol noktaları</div>
    <div><b>Süre:</b> ~4 dakika</div>
    <div><b>Çıktı:</b> PDF panelleri + ham diziler (NPZ)</div>
    <div><b>Geometri:</b> 3,50 in / 7,16 in kolon</div>
  </div>
  <div style="margin-top:20px;padding:13px 16px;background:rgba(56,139,253,.10);border-left:4px solid #388bfd;border-radius:4px;font-size:.86em;line-height:1.55;">
    <b>Neden gerekli:</b> Önceki notebook'ların ürettiği figürler 160 dpi'dır; IEEE renkli/gri görseller için en az 300 dpi, çizgi grafikleri için 600 dpi ister. Grafikler kayıtlı verilerden yerel olarak vektör PDF'e dönüştürülebilir; ancak <b>ham radyografi içeren paneller</b> için görüntülere erişim gerekir. Bu notebook o panelleri üretir ve ayrıca tüm ara dizileri kaydeder; böylece yerleşim yerelde yeniden düzenlenebilir, notebook'u tekrar çalıştırmaya gerek kalmaz.
  </div>
</div>

## Tasarım kuralları

Şekiller doğrudan **son boyutlarında** üretilir; LaTeX içinde ölçeklenmezler. Ölçekleme,
yazı boyutlarını bozduğu ve IEEE'nin en sık geri döndürdüğü biçim hatası olduğu için
kaçınılmıştır.

| Parametre | Değer | Gerekçe |
| :---- | :---- | :---- |
| Tek kolon genişliği | 3,50 in (88,9 mm) | IEEE iki kolon yerleşimi |
| Çift kolon genişliği | 7,16 in (181,9 mm) | IEEE iki kolon yerleşimi |
| Yazı tipi | DejaVu Sans | Hem Kaggle hem yerelde mevcut — figürler arası tutarlılık |
| Etiket / eksen | 8 pt / 7 pt | Basılı boyutta okunur, gövde metniyle uyumlu |
| Çıktı biçimi | PDF (vektör) | Metin ve çizgiler vektör; gömülü raster 600 dpi |
| Panel harfleri | (a), (b), … kalın | IEEE alt şekil geleneği |

### Üretilen paneller

1. **`fig1_pipeline.pdf`** — ön-işleme zinciri ve üç ablasyon kolunun girdileri (çift kolon)
2. **`fig_occlusion_conditions.pdf`** — oklüzyon bölgeleri (çift kolon, ek malzeme adayı)
3. **`fig_attention_examples.pdf`** — örnek dikkat bindirmeleri (çift kolon, ek malzeme adayı)
4. **`paper_panel_arrays.npz`** — tüm ara diziler; yerleşim yerelde yeniden kurulabilir

### Girdiler

| # | Sekme | Kimlik |
| :---: | :---- | :---- |
| 1 | Datasets | `yusufmurtaza01/chest-xray-pneumonia-balanced-dataset` |
| 2 | **Notebooks** | `segmentation-ablation-lung-focused-vit` (kontrol noktaları) |
| 3 | Competitions | `rsna-pneumonia-detection-challenge` |

GPU ve Internet açık olmalıdır.

In [ ]:
import os, re, io, glob, json, time, random, zipfile, warnings, types
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms, models

# ---------------- IEEE GEOMETRISI (sabit) ----------------
COL1, COL2 = 3.50, 7.16          # inch
FS_LABEL, FS_TICK, FS_PANEL = 8, 7, 9
SEED = 42
# ---------------------------------------------------------

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans"],
    "font.size": FS_LABEL,
    "axes.labelsize": FS_LABEL, "axes.titlesize": FS_LABEL,
    "xtick.labelsize": FS_TICK, "ytick.labelsize": FS_TICK,
    "legend.fontsize": FS_TICK,
    "axes.linewidth": 0.6, "lines.linewidth": 1.0,
    "xtick.major.width": 0.6, "ytick.major.width": 0.6,
    "figure.facecolor": "white", "savefig.facecolor": "white",
    "savefig.bbox": "tight", "savefig.pad_inches": 0.01,
    "pdf.fonttype": 42,            # TrueType gomme - IEEE gereksinimi
    "ps.fonttype": 42,
})

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WORK = "/kaggle/working"
ARMS = ["raw", "roi", "lung"]
ARM_TITLE = {"raw": "(A) No segmentation", "roi": "(B) ROI crop only",
             "lung": "(C) Mask + ROI crop"}
print(f"Cihaz: {device} | tek kolon {COL1}\" | cift kolon {COL2}\"")

In [ ]:
# ── Girdiler ─────────────────────────────────────────────────────────────
INPUT = "/kaggle/input"

def find_classification_root(root=INPUT):
    hits = []
    for r, dirs, _ in os.walk(root):
        if os.path.basename(r) == "train" and {"NORMAL", "PNEUMONIA"} <= set(dirs):
            hits.append(os.path.dirname(r))
    hits.sort(key=lambda p: (0 if "balanced" in p.lower() else 1, len(p)))
    return hits[0] if hits else None

def find_dir_with(fname, root=INPUT):
    for r, _, files in os.walk(root):
        if fname in files:
            return r
    return None

BASE = find_classification_root()
RSNA_BASE = find_dir_with("stage_2_detailed_class_info.csv")
ckpts = {}
for p in glob.glob(os.path.join(INPUT, "**", "*.pth"), recursive=True):
    b = os.path.basename(p).lower()
    for a in ARMS:
        if f"_{a}." in b:
            ckpts[a] = p
assert BASE is not None, "Siniflandirma veri kumesi bulunamadi."
assert len(ckpts) == 3, f"Kontrol noktasi eksik: {set(ARMS) - set(ckpts)}"
print("veri kumesi:", BASE)
print("RSNA       :", RSNA_BASE)
for a in ARMS:
    print(f"  ckpt {a:<5}: {os.path.basename(ckpts[a])}")

In [ ]:
# ── Modeller ve segmentasyon ─────────────────────────────────────────────
def build_vit_inference(nc, dp):
    m = models.vit_b_16(weights=None)
    m.heads.head = nn.Sequential(nn.Dropout(dp), nn.Linear(m.heads.head.in_features, nc))
    return m

MODELS, CFG, CLASS_TO_IDX = {}, None, None
for a in ARMS:
    ck = torch.load(ckpts[a], map_location=device, weights_only=False)
    CFG, CLASS_TO_IDX = ck["config"], ck["class_to_idx"]
    m = build_vit_inference(CFG["num_classes"], CFG.get("dropout", 0.1)).to(device)
    m.load_state_dict(ck["model_state_dict"], strict=True)
    MODELS[a] = m.eval()
PNEU_IDX = CLASS_TO_IDX["PNEUMONIA"]
S = CFG["img_size"]
FILL = int(round(CFG["mean"][0] * 255))
eval_tf = transforms.Compose([transforms.Resize((S, S)), transforms.ToTensor(),
                              transforms.Normalize(CFG["mean"], CFG["std"])])
print(f"Modeller yuklendi | girdi {S} | dolgu {FILL}")

import transformers
from transformers import AutoModel
def _load_seg():
    return AutoModel.from_pretrained("ianpan/chest-x-ray-basic",
                                     trust_remote_code=True).to(device).eval()
_of = getattr(transformers.modeling_utils.PreTrainedModel, "_finalize_model_loading", None)
try:
    if _of is not None:
        def _sf(model, *a, **k):
            if not hasattr(model, "all_tied_weights_keys"):
                model.all_tied_weights_keys = {}
            return _of(model, *a, **k)
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _sf
    seg_model = _load_seg()
except Exception as e:
    if "all_tied_weights_keys" in str(e):
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = {}
        seg_model = _load_seg()
    else:
        raise
finally:
    if _of is not None:
        transformers.modeling_utils.PreTrainedModel._finalize_model_loading = _of
print("Segmentasyon modeli hazir.")

In [ ]:
# ── On-isleme (ablasyonla birebir ayni) ──────────────────────────────────
def load_image_any(path, short_max):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".dcm":
        import pydicom
        d = pydicom.dcmread(path)
        arr = d.pixel_array.astype(np.float32); arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        if getattr(d, "PhotometricInterpretation", "MONOCHROME2") == "MONOCHROME1":
            arr = 1.0 - arr
        pil = Image.fromarray((arr * 255).astype(np.uint8)).convert("RGB")
    else:
        pil = Image.open(path).convert("RGB")
    W0, H0 = pil.size; sh = min(W0, H0)
    if sh > short_max:
        s = short_max / sh
        pil = pil.resize((int(round(W0 * s)), int(round(H0 * s))), Image.BILINEAR)
    return np.asarray(pil).astype(np.uint8), np.asarray(pil.convert("L"))

@torch.inference_mode()
def lung_mask_ianpan(gray, out_hw):
    x = seg_model.preprocess(gray)
    x = torch.from_numpy(x).unsqueeze(0).unsqueeze(0).float().to(device)
    lg = F.interpolate(seg_model(x)["mask"], size=out_hw, mode="bilinear", align_corners=False)
    p = lg.argmax(1)[0].cpu().numpy()
    return ((p == 1) | (p == 2)).astype(np.uint8)

def prep_all(rgb, lung, cfg):
    '''Uc kolun girdisi + maskeli tam kare + RoI kutusu.'''
    H, W = lung.shape; short = min(H, W); Sz = cfg["img_size"]
    out = {"raw": cv2.resize(rgb, (Sz, Sz), interpolation=cv2.INTER_AREA)}
    dil = max(1, int(round(short * cfg["mask_dilate_frac"])))
    kern = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * dil + 1, 2 * dil + 1))
    mask_d = cv2.dilate(lung, kern, 1)
    soft = mask_d.astype(np.float32)
    f = cfg["mask_feather"]
    if f and f >= 3:
        if f % 2 == 0:
            f += 1
        soft = cv2.GaussianBlur(soft, (f, f), 0)
    soft = np.clip(soft, 0, 1)[..., None]
    fill = np.array([m * 255.0 for m in cfg["mean"]], np.float32)
    masked_full = (rgb.astype(np.float32) * soft + fill * (1 - soft)).clip(0, 255).astype(np.uint8)
    ys, xs = np.where(mask_d > 0)
    y0, y1 = int(ys.min()), int(ys.max()); x0, x1 = int(xs.min()), int(xs.max())
    pad = int(round(short * cfg["roi_pad_frac"]))
    y0 = max(0, y0 - pad); x0 = max(0, x0 - pad)
    y1 = min(H - 1, y1 + pad); x1 = min(W - 1, x1 + pad)
    bh, bw = y1 - y0 + 1, x1 - x0 + 1
    side = min(max(bh, bw), min(H, W))
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    ty0 = max(0, cy - side // 2); tx0 = max(0, cx - side // 2)
    ty1 = min(H, ty0 + side); tx1 = min(W, tx0 + side)
    ty0 = max(0, ty1 - side); tx0 = max(0, tx1 - side)
    out["roi"] = cv2.resize(rgb[ty0:ty1, tx0:tx1], (Sz, Sz), interpolation=cv2.INTER_AREA)
    out["lung"] = cv2.resize(masked_full[ty0:ty1, tx0:tx1], (Sz, Sz), interpolation=cv2.INTER_AREA)
    m224 = cv2.resize(mask_d[ty0:ty1, tx0:tx1], (Sz, Sz), interpolation=cv2.INTER_NEAREST)
    return out, masked_full, mask_d, (tx0, ty0, side), m224.astype(np.uint8)

print("On-isleme hatti hazir.")

In [ ]:
# ── Ornek goruntulerin secimi (deterministik) ────────────────────────────
def pick(split, cls, k):
    '''Ornek secimi. Augmentasyon kopyalari (_aug_) DISLANIR: makale
    augmentasyon kaynakli sizintiyi raporladigindan, vitrin ornegi olarak
    ozgun bir goruntu kullanilmalidir.'''
    d = os.path.join(BASE, split, cls)
    fs = sorted(f for f in os.listdir(d)
                if f.lower().endswith((".jpeg", ".jpg", ".png"))
                and not re.search(r"_aug_\d+", f))
    rng = random.Random(SEED)
    rng.shuffle(fs)
    return [os.path.join(d, f) for f in fs[:k]]

EX = [("NORMAL", pick("val", "NORMAL", 1)[0]),
      ("PNEUMONIA", pick("val", "PNEUMONIA", 1)[0])]
print("Secilen ornekler:")
for c, p in EX:
    b = os.path.basename(p)
    tag = "  <-- AUGMENTASYON KOPYASI, DEGISTIR" if re.search(r"_aug_\d+", b) else "  (ozgun)"
    print(f"  {c:<10} {b}{tag}")

DATA = {}
for cls, path in EX:
    rgb, gray = load_image_any(path, CFG["orig_short_max"])
    lung = lung_mask_ianpan(gray, gray.shape)
    arms, masked_full, mask_d, box, m224 = prep_all(rgb, lung, CFG)
    DATA[cls] = {"rgb": rgb, "lung": lung, "mask_d": mask_d, "masked_full": masked_full,
                 "box": box, "m224": m224, **{f"arm_{a}": arms[a] for a in ARMS}}
    print(f"  {cls}: orijinal {rgb.shape[:2]}  akciger alani %{100*lung.mean():.1f}")

## Şekil 1 — Ön-işleme zinciri ve üç kolun girdileri

Çift kolon genişliğinde, iki satır (bir NORMAL, bir PNEUMONIA) ve beş sütun:
orijinal radyograf, akciğer konturu bindirilmiş hâl, ve üç kolun nihai $224\times224$
girdisi. Kolon başlıkları yalnızca üst satırda verilir; satır etiketleri sol kenarda
döndürülmüş olarak yerleştirilir.

In [ ]:
def imshow_clean(ax, img, cmap=None):
    ax.imshow(img, cmap=cmap, interpolation="lanczos")
    ax.set_xticks([]); ax.set_yticks([])
    # contour/patch eklendiginde eksen sinirlarinin buyumesini engelle
    h, w = img.shape[:2]
    ax.set_xlim(-0.5, w - 0.5); ax.set_ylim(h - 0.5, -0.5)
    ax.set_autoscale_on(False)
    for s in ax.spines.values():
        s.set_linewidth(0.5); s.set_color("#666666")

NCOL = 5
fig, axes = plt.subplots(2, NCOL, figsize=(COL2, COL2 * 2 / NCOL * 1.04))
COLTITLE = ["Input radiograph", "Lung segmentation",
            ARM_TITLE["raw"], ARM_TITLE["roi"], ARM_TITLE["lung"]]

for r, (cls, _) in enumerate(EX):
    D = DATA[cls]
    imshow_clean(axes[r, 0], D["rgb"])
    imshow_clean(axes[r, 1], D["rgb"])
    axes[r, 1].contour(D["lung"], levels=[0.5], colors="#0D8FA2", linewidths=0.8)
    x0, y0, sd = D["box"]
    axes[r, 1].add_patch(plt.Rectangle((x0, y0), sd, sd, fill=False,
                                       ec="#C2900A", lw=0.7, ls=(0, (3, 2))))
    for c, a in enumerate(ARMS, start=2):
        imshow_clean(axes[r, c], D[f"arm_{a}"])
    axes[r, 0].set_ylabel(cls.capitalize(), fontsize=FS_LABEL, labelpad=2)

for c in range(NCOL):
    axes[0, c].set_title(COLTITLE[c], fontsize=FS_TICK, pad=3)

plt.subplots_adjust(wspace=0.04, hspace=0.04)
fig.savefig(os.path.join(WORK, "fig1_pipeline.pdf"), dpi=600)
fig.savefig(os.path.join(WORK, "fig1_pipeline.png"), dpi=600)
plt.show()
print("fig1_pipeline.pdf / .png yazildi")

## Oklüzyon koşulları paneli

Yöntem bölümünde bölge tanımlarını göstermek için; sayfa sınırı nedeniyle ek malzemeye
alınabilir. Bölgeler ablasyon notebook'undaki tanımların birebir aynısıdır.

In [ ]:
CORNER_F, BORDER_F, SUBDIA_F, UPPER_F, CENTER_F = 0.22, 0.08, 0.28, 0.20, 0.20

def geom_masks(Sz):
    m = {}
    c = int(round(Sz * CORNER_F)); a = np.zeros((Sz, Sz), bool)
    a[:c, :c] = a[:c, -c:] = a[-c:, :c] = a[-c:, -c:] = True; m["Corners"] = a
    b = int(round(Sz * BORDER_F)); a = np.zeros((Sz, Sz), bool)
    a[:b, :] = a[-b:, :] = a[:, :b] = a[:, -b:] = True; m["Border"] = a
    a = np.zeros((Sz, Sz), bool); a[int(round(Sz * (1 - SUBDIA_F))):, :] = True
    m["Subdiaphragm"] = a
    a = np.zeros((Sz, Sz), bool); a[:int(round(Sz * UPPER_F)), :] = True; m["Upper"] = a
    a = np.zeros((Sz, Sz), bool)
    lo = int(round(Sz * (0.5 - CENTER_F / 2))); hi = int(round(Sz * (0.5 + CENTER_F / 2)))
    a[:, lo:hi] = True; m["Centre"] = a
    return m

GEOM = geom_masks(S)
D = DATA["PNEUMONIA"]
base = D["arm_raw"]; lungm = cv2.resize(D["lung"], (S, S), interpolation=cv2.INTER_NEAREST).astype(bool)
rng = np.random.default_rng(SEED)

def rand_rect(Sz, af):
    t = af * Sz * Sz
    ar = np.exp(rng.uniform(np.log(0.5), np.log(2.0)))
    h = int(np.clip(round(np.sqrt(t / ar)), 1, Sz)); w = int(np.clip(round(t / max(h, 1)), 1, Sz))
    y = rng.integers(0, Sz - h + 1); x = rng.integers(0, Sz - w + 1)
    a = np.zeros((Sz, Sz), bool); a[y:y + h, x:x + w] = True
    return a

CONDS = [("Unoccluded", np.zeros((S, S), bool)),
         ("Lungs", lungm), ("Outside lungs", ~lungm)] + list(GEOM.items()) + \
        [("Random (area-matched)", rand_rect(S, lungm.mean()))]

n = len(CONDS)
fig, axes = plt.subplots(1, n, figsize=(COL2, COL2 / n * 1.20))
for ax, (name, mk) in zip(axes, CONDS):
    v = base.copy(); v[mk] = FILL
    imshow_clean(ax, v)
    ax.set_title(f"{name}\n{100*mk.mean():.0f}%", fontsize=6.4, pad=2, linespacing=1.15)
plt.subplots_adjust(wspace=0.05)
fig.savefig(os.path.join(WORK, "fig_occlusion_conditions.pdf"), dpi=600)
fig.savefig(os.path.join(WORK, "fig_occlusion_conditions.png"), dpi=600)
plt.show()
print("fig_occlusion_conditions.pdf / .png yazildi")

## Örnek dikkat bindirmeleri

Toplu dikkat haritaları kayıtlı dizilerden yerel olarak üretilebildiğinden burada
yalnızca **tekil örnekler** hazırlanır; bunlar ek malzemede modelin tipik davranışını
göstermek için kullanılabilir.

In [ ]:
class Rollout:
    def __init__(self, model):
        self.m = model; self.maps = []; self.h = []
        for layer in model.encoder.layers:
            sa = layer.self_attention; orig = sa.forward
            def mk(o):
                def p(self_mod, *a, **kw):
                    kw["need_weights"] = True; kw["average_attn_weights"] = False
                    return o(*a, **kw)
                return p
            sa.forward = types.MethodType(mk(orig), sa)
            def hk(obj):
                def f(m_, i_, o_):
                    if isinstance(o_, tuple) and len(o_) > 1 and o_[1] is not None:
                        obj.maps.append(o_[1].detach().cpu())
                return f
            self.h.append(sa.register_forward_hook(hk(self)))
    def remove(self):
        for h in self.h:
            h.remove()
    def __call__(self, x):
        self.maps = []; self.m.eval()
        with torch.no_grad():
            out = self.m(x.unsqueeze(0).to(device))
        prob = torch.softmax(out, 1)[0, PNEU_IDX].item()
        N = self.maps[0].size(-1); R = torch.eye(N)
        for A in self.maps:
            a = A.squeeze(0)
            if a.dim() == 3:
                a = a.mean(0)
            a = a + torch.eye(N); a = a / (a.sum(-1, keepdim=True) + 1e-8)
            R = torch.matmul(a, R)
        v = R[0, 1:].numpy(); k = int(round(v.size ** 0.5))
        v = v.reshape(k, k); v = (v - v.min()) / (v.max() - v.min() + 1e-8)
        v = cv2.resize(v.astype(np.float32), (S, S))
        return (v - v.min()) / (v.max() - v.min() + 1e-8), prob

CMAP = plt.matplotlib.colors.LinearSegmentedColormap.from_list(
    "att", ["#0b1a33", "#0f4c81", "#2a9d8f", "#e9c46a", "#e76f51"], N=256)

fig, axes = plt.subplots(2, 4, figsize=(COL2 * 0.72, COL2 * 0.72 * 2 / 4 * 1.10))
ATT = {}
for r, (cls, _) in enumerate(EX):
    D = DATA[cls]
    imshow_clean(axes[r, 0], D["arm_raw"])
    axes[r, 0].set_ylabel(cls.capitalize(), fontsize=FS_LABEL, labelpad=2)
    for c, a in enumerate(ARMS, start=1):
        ro = Rollout(MODELS[a])
        sal, prob = ro(eval_tf(Image.fromarray(D[f"arm_{a}"])))
        ro.remove()
        ATT[(cls, a)] = sal
        img = D[f"arm_{a}"].astype(np.float32) / 255.0
        ov = 0.55 * img + 0.45 * CMAP(sal)[:, :, :3]
        imshow_clean(axes[r, c], np.clip(ov, 0, 1))
for c, t in enumerate(["Model input"] + [ARM_TITLE[a] for a in ARMS]):
    axes[0, c].set_title(t, fontsize=FS_TICK, pad=3)
plt.subplots_adjust(wspace=0.04, hspace=0.04)
fig.savefig(os.path.join(WORK, "fig_attention_examples.pdf"), dpi=600)
fig.savefig(os.path.join(WORK, "fig_attention_examples.png"), dpi=600)
plt.show()
print("fig_attention_examples.pdf / .png yazildi")

In [ ]:
# ── Ham dizilerin kaydi: yerleşim yerelde yeniden kurulabilsin ───────────
save = {}
for cls, _ in EX:
    D = DATA[cls]
    save[f"{cls}__rgb"] = D["rgb"]
    save[f"{cls}__lung"] = D["lung"]
    save[f"{cls}__mask_d"] = D["mask_d"]
    save[f"{cls}__masked_full"] = D["masked_full"]
    save[f"{cls}__box"] = np.array(D["box"])
    save[f"{cls}__m224"] = D["m224"]
    for a in ARMS:
        save[f"{cls}__arm_{a}"] = D[f"arm_{a}"]
        if (cls, a) in ATT:
            save[f"{cls}__att_{a}"] = ATT[(cls, a)].astype(np.float32)
for name, mk in CONDS:
    save[f"cond__{name.replace(' ', '_').replace('(', '').replace(')', '')}"] = mk

np.savez_compressed(os.path.join(WORK, "paper_panel_arrays.npz"), **save)

print("Kaydedilenler:")
for f in sorted(os.listdir(WORK)):
    if f.startswith(("fig1_", "fig_", "paper_panel")):
        print(f"  {f:<36} {os.path.getsize(os.path.join(WORK, f))/1e6:>7.2f} MB")
print("\nNot: PDF'ler IEEE kolon genisliginde uretildi; LaTeX icinde"
      " \\includegraphics[width=\\textwidth]{...} ile OLCEKLENMEDEN kullanilmalidir.")

## Kullanım

`fig1_pipeline.pdf` doğrudan çift kolon şekil olarak yerleştirilir:

```latex
\begin{figure*}[t]
  \centering
  \includegraphics[width=\textwidth]{fig1_pipeline.pdf}
  \caption{...}
  \label{fig:pipeline}
\end{figure*}
```

Şekil zaten 7,16 inç genişliğinde üretildiği için `width=\textwidth` ölçekleme yapmaz;
yazı boyutları basılı hâlde tasarlandığı gibi 7-8 pt çıkar.

`paper_panel_arrays.npz` dosyası tüm ara dizileri içerir. Yerleşimde değişiklik
gerekirse (panel sayısı, sıralama, etiketler) notebook'u yeniden çalıştırmaya gerek
yoktur; paneller bu dosyadan yerel olarak yeniden üretilebilir.